# Module 01: LLM basics + the AI gateway (Colab)

Read the chapter first. This notebook:
1. starts the **LiteLLM gateway** as a background process. **Only that process receives your vendor keys.**
2. runs each lesson, in the same order as the chapter.

Secrets needed (key icon, left sidebar): `ANTHROPIC_API_KEY`, `OPENAI_API_KEY`, `LITELLM_MASTER_KEY`, `DATABASE_URL`. After lesson L04 you will add one more: `DOCUMIND_API_KEY`.

Set `USE_MOCK = True` to practise for free with canned replies (no vendor keys needed).

In [ ]:
# ── Get the course code ──────────────────────────────────────────────────────
# BRANCH: "main" once the course is merged; otherwise the branch you are studying.
REPO = "github.com/panwars87/sha-genai-learning.git"
BRANCH = "main"

import os, subprocess
from google.colab import userdata

def secret(name, required=True):
    """Read a Colab Secret by name (never printed)."""
    try:
        return userdata.get(name)
    except Exception:
        if required:
            raise SystemExit(f"Add '{name}' in Colab Secrets (key icon, left sidebar) and allow notebook access.")
        return None

# A private repo needs a GitHub token secret named GITHUB_TOKEN (read-only is enough).
token = secret("GITHUB_TOKEN", required=False)
url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
if not os.path.isdir("/content/sha-genai-learning"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, "/content/sha-genai-learning"], check=True)
%cd /content/sha-genai-learning
!pip -q install uv
print("Ready.")

## Start the AI gateway

In [ ]:
USE_MOCK = False
MODULE_DIR = "modules/01-llm-basics-and-ai-gateway"

import time, urllib.request

# Vendor keys go ONLY into the gateway process environment, not into this notebook's.
gateway_env = dict(os.environ)
for name in ["LITELLM_MASTER_KEY", "DATABASE_URL"]:
    gateway_env[name] = secret(name)
for name in ["ANTHROPIC_API_KEY", "OPENAI_API_KEY"]:
    value = secret(name, required=not USE_MOCK)
    if value:
        gateway_env[name] = value

config = f"{MODULE_DIR}/gateway/config{'.mock' if USE_MOCK else ''}.yaml"
log = open("/content/gateway.log", "w")
gateway = subprocess.Popen(["bash", "infra/start_gateway.sh", config],
                           env=gateway_env, stdout=log, stderr=subprocess.STDOUT)

for _ in range(150):  # first start installs LiteLLM and sets up the database (~1-2 min)
    try:
        urllib.request.urlopen("http://localhost:4000/health/liveliness", timeout=2)
        print("Gateway is up at http://localhost:4000")
        break
    except Exception:
        if gateway.poll() is not None:
            raise SystemExit("Gateway stopped. Read /content/gateway.log")
        time.sleep(2)

## App settings for this notebook
The app and lessons get only the admin key (for L04) and, once you have it, DocuMind's virtual key.

In [ ]:
os.environ["DOCUMIND_RUNTIME"] = "colab"
os.environ["LITELLM_MASTER_KEY"] = secret("LITELLM_MASTER_KEY")   # admin lessons (L03, L04, E01-E04)
virtual_key = secret("DOCUMIND_API_KEY", required=False)
if virtual_key:
    os.environ["DOCUMIND_API_KEY"] = virtual_key
    print("DOCUMIND_API_KEY loaded.")
else:
    print("No DOCUMIND_API_KEY yet. You will create it in L04.")
!cd modules/01-llm-basics-and-ai-gateway/app && uv sync -q

## L01 · A direct call, the old way
The one deliberate exception: this lesson calls Anthropic directly, so it alone gets the vendor key, for this single command.

In [ ]:
subprocess.run(["uv", "run", "-q", "--project", "app", "python", "lessons/L01_direct_call_the_old_way.py"],
               cwd=MODULE_DIR, env={**os.environ, "ANTHROPIC_API_KEY": secret("ANTHROPIC_API_KEY")})

## L03 · Meet the gateway

In [ ]:
!cd modules/01-llm-basics-and-ai-gateway && uv run -q --project app python lessons/L03_gateway_health.py

## L04 · Issue DocuMind's virtual key
Copy the printed key into Colab Secrets as **`DOCUMIND_API_KEY`**, then re-run the *App settings* cell above.
Lost it? Change the command to add `--replace`.

In [ ]:
!cd modules/01-llm-basics-and-ai-gateway && uv run -q --project app python lessons/L04_create_virtual_key.py

## L05 · One client, two vendors

In [ ]:
!cd modules/01-llm-basics-and-ai-gateway && uv run -q --project app python lessons/L05_one_client_two_vendors.py

## L06 · Tokens, cost and knobs

In [ ]:
!cd modules/01-llm-basics-and-ai-gateway && uv run -q --project app python lessons/L06_tokens_cost_and_knobs.py

## L07 · No memory

In [ ]:
!cd modules/01-llm-basics-and-ai-gateway && uv run -q --project app python lessons/L07_statelessness_and_memory.py

## Break it

In [ ]:
!cd modules/01-llm-basics-and-ai-gateway && uv run -q --project app python experiments/E01_revoked_key.py
!cd modules/01-llm-basics-and-ai-gateway && uv run -q --project app python experiments/E02_budget_exceeded.py
!cd modules/01-llm-basics-and-ai-gateway && uv run -q --project app python experiments/E03_answer_cut_off.py
!cd modules/01-llm-basics-and-ai-gateway && uv run -q --project app python experiments/E04_model_not_allowed.py

## The app: `documind summarize`

In [ ]:
!cd modules/01-llm-basics-and-ai-gateway && uv run -q --project app documind summarize ../../data/invoices/INV-1001-acme-supplies.txt
!cd modules/01-llm-basics-and-ai-gateway && uv run -q --project app documind summarize ../../data/receipts/RCPT-3002-handy-hardware.txt

## Stop the gateway

In [ ]:
gateway.terminate(); print('Gateway stopped.')